<a href="https://colab.research.google.com/github/ambreenraheem/PGD_generative_AI_NED/blob/main/NGO_assistant_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Ambreen Abdul Raheem
Power BI Data Analyst (Upwork Freelancer)\
This is my Pakistan Based NGO,s AI Assistant, who can provide any kind of information from these NGO's you need, like: INDUS HOPITAL, CHIPPA, SHUKAT KHANUM CANCER HOSPITAL, TCF and will upgrade and add more NGO's information soon.

In [ ]:

import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List, Optional, Dict
import chainlit as cl

# === Your agents framework (assumed available) ===
from agents import (
    Agent,
    handoff,
    InputGuardrail,
    GuardrailFunctionOutput,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    function_tool,
)
from agents.exceptions import InputGuardrailTripwireTriggered


# Setup
load_dotenv()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

# Disable any tracing/telemetry by default
set_tracing_disabled(disabled=True)

# Configure Gemini via OpenAI-compatible endpoint
external_client: AsyncOpenAI = AsyncOpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
llm_model: OpenAIChatCompletionsModel = OpenAIChatCompletionsModel(
    model="gemini-2.5-flash",
    openai_client=external_client,
)


# Domain models
class Volunteering(BaseModel):
    roles: List[str]
    process: str

class NGO(BaseModel):
    name: str
    type: str
    programs: List[str]
    services: List[str]
    official_links: List[str]
    notes: Optional[str] = None
    volunteering: Optional[Volunteering] = None


# Knowledge base (quick-start)
NGO_DATA: Dict[str, NGO] = {
    "tcf": NGO(
        name="The Citizens Foundation (TCF)",
        type="Education",
        programs=[
            "TCF Schools network (primary–secondary)",
            "Scholarships & higher-education support",
            "Teacher training & curriculum development",
        ],
        services=[
            "Low-cost quality schooling in underserved areas",
            "Student sponsorship & alumni support",
        ],
        official_links=["https://www.tcf.org.pk/"],
        notes="For admissions/scholarships, see official site pages; verify current deadlines.",
    ),
    "chhipa": NGO(
        name="Chhipa Welfare Association",
        type="Emergency Relief & Welfare",
        programs=[
            "Ambulance & emergency response",
            "Ration & relief distribution",
            "Blood donation drives",
        ],
        services=[
            "24/7 ambulance service",
            "Disaster/relief operations",
        ],
        official_links=["https://www.chhipa.org/"],
        notes="Confirm current volunteer intake and training schedule on the official site.",
        volunteering=Volunteering(
            roles=["Ambulance support", "Relief camp assistance", "Logistics & distribution"],
            process="Apply via official website form or contact volunteer desk; bring CNIC at onboarding.",
        ),
    ),
    "saylani": NGO(
        name="Saylani Welfare International Trust",
        type="Multi-sector Welfare",
        programs=[
            "IT & digital skills training",
            "Food/ration distribution",
            "Medical & social welfare",
        ],
        services=[
            "Daily meals program",
            "Skills training (e.g., IT)",
            "Medical clinics & camps",
        ],
        official_links=["https://www.saylaniwelfare.com/"],
        notes="Program availability varies by city; check current intakes on the website.",
    ),
    "akhuwat": NGO(
        name="Akhuwat",
        type="Livelihoods & Education Support",
        programs=[
            "Interest-free microfinance",
            "Entrepreneurship & micro-grants",
            "Education initiatives (e.g., Akhuwat College)",
        ],
        services=[
            "Qarz-e-Hasna loans",
            "Business advisory & training",
        ],
        official_links=["https://www.akhuwat.org.pk/", "https://akhuwat.org/"],
        notes="Verify branch schedules and current loan application windows.",
    ),
    "shaukat_khanum": NGO(
        name="Shaukat Khanum Memorial Cancer Hospital & Research Centre",
        type="Health (Oncology)",
        programs=[
            "Cancer screening & diagnostics",
            "Treatment & patient support",
            "Awareness campaigns",
        ],
        services=[
            "OPD & IPD services",
            "Financial support for eligible patients",
        ],
        official_links=["https://shaukatkhanum.org.pk/"],
        notes="For appointments and eligibility, follow hospital instructions only.",
    ),
    "indus": NGO(
        name="Indus Hospital & Health Network (IHHN)",
        type="Health",
        programs=[
            "Free-of-cost hospital & community health services",
            "Screening & vaccination drives",
            "Public health outreach",
        ],
        services=[
            "OPD/IPD at partner sites",
            "Community programs",
        ],
        official_links=["https://indushospital.org.pk/"],
        notes="Services vary by location; check the network’s site for city-wise availability.",
    ),
}

# Simple lookup (so program agents can fetch vetted info)
@function_tool
def ngo_lookup(name: str) -> dict:
    """Look up a known NGO by key or alias (e.g., "tcf", "saylani"). Returns a dict for LLM use."""
    key = (name or "").strip().lower().replace(" ", "_")
    if key in NGO_DATA:
        return NGO_DATA[key].model_dump()
    # basic aliasing
    aliases = {
        "the_citizens_foundation": "tcf",
        "the citizens foundation": "tcf",
        "shaukat": "shaukat_khanum",
        "ihhn": "indus",
    }
    if key in aliases and aliases[key] in NGO_DATA:
        return NGO_DATA[aliases[key]].model_dump()
    # fallback: best-effort contains search
    for k, v in NGO_DATA.items():
        if key and (key in k or key in v.name.lower()):
            return v.model_dump()
    return {"error": "NGO not found. Try one of: " + ", ".join(sorted(NGO_DATA.keys()))}


# Guardrail Output Schema
class NGOInputCheck(BaseModel):
    """Guardrail classification for NGO assistant safety & scope checks."""
    topic: str = Field(
        ...,
        description=(
            "The main category of the user's query. "
            "One of: education, health, relief, livelihood, protection, volunteering, complaints, general"
        ),
    )
    collect_personal_data: bool = Field(
        ...,
        description=(
            "True if the user is trying to share or requesting to store "
            "personal identifiers (CNIC/NIC, phone, exact address) "
            "without a proper consent workflow."
        ),
    )
    medical_diagnosis: bool = Field(
        ...,
        description=(
            "True if the user asks for a medical diagnosis or prescription "
            "beyond general health info and safe guidance."
        ),
    )
    legal_advice: bool = Field(
        ...,
        description=(
            "True if the user asks for binding legal advice beyond general awareness."
        ),
    )
    unsafe_request: bool = Field(
        ...,
        description=(
            "True if the message contains violent, hateful, self-harm, or otherwise unsafe content."
        ),
    )
    reasoning: str = Field(
        ...,
        description=(
            "Brief reasoning explaining why the above flags and topic were chosen. "
            "Keep concise and professional."
        ),
    )


# Agents
# Guardrail classifier agent
guardrail_agent = Agent(
    name="NGO's AI Assistant",
    instructions=(
        "You are the safety and scope classifier for an NGO assistant. "
        "Given a user query, identify the most relevant NGO topic and whether the message "
        "contains restricted intents.\n\n"
        "TOPICS: education, health, relief, livelihood, protection, volunteering, complaints, general.\n"
        "Flags:\n"
        "- collect_personal_data: true if user wants to give or asks you to store personal identifiers (CNIC/NIC, phone numbers, exact addresses) without consent workflow.\n"
        "- medical_diagnosis: true if user requests diagnosis or prescription (beyond general info and 'see a doctor' guidance).\n"
        "- legal_advice: true if user requests binding legal advice.\n"
        "- unsafe_request: true for violent, hateful, self-harm, or otherwise disallowed content.\n"
        "Respond with the NGOInputCheck schema only."
    ),
    output_type=NGOInputCheck,
    model=llm_model,
)

# Program agents
education_agent = Agent(
    name="Education Program Agent",
    handoff_description="Scholarships, school enrollment, after-school tutoring, learning resources.",
    instructions=(
        "You are an NGO Education Program officer in Pakistan. Answer only about education: "
        "scholarships, enrollment, literacy classes, teacher training, school supplies. "
        "Reference well-known orgs where relevant: The Citizens Foundation (TCF) for low-cost quality schooling "
        "and scholarships; Saylani for IT/skills courses; Akhuwat College/education support initiatives for merit-based aid. "
        "When asked for process, provide clear steps, typical documents (CNIC/B-Form, income proof), and city-wise offices if known.\n\nUse this structured response style in every answer:\n"
        "1) Overview (1–2 lines about the NGO/service relevant to the question)\n"
        "2) Eligibility/Who it helps (bullet points)\n"
        "3) Step-by-step Process (numbered steps, include typical documents like CNIC/B-Form if relevant)\n"
        "4) Where/When (city-wise or general availability; mention 'check official site for latest')\n"
        "5) Official Links (list from the knowledge base via ngo_lookup)\n"
        "6) Next Steps (what the user should do now)\n\n"
        "When the user names an organization (e.g., TCF, Chhipa, Saylani, Akhuwat, Shaukat Khanum, Indus),"
        "call the ngo_lookup tool with that name and weave the returned data into the response."
        "If info is missing, say so briefly and advise checking official links. If outside scope, defer."
    ),
    tools=[ngo_lookup],
    model=llm_model,
)

health_agent = Agent(
    name="Health Program Agent",
    handoff_description="Medical camps, vaccination drives, health awareness sessions.",
    instructions=(
        "You are an NGO Health Program officer in Pakistan. Share general health camp information, vaccination schedules, "
        "screening camps, and referral pathways. Mention major providers where relevant: Shaukat Khanum (cancer screening/support), "
        "Indus Hospital & Health Network (free treatment and community health), and Saylani medical services. "
        "Do NOT provide diagnosis or prescribe medications. Always recommend visiting qualified clinicians and official OPD desks."
    ),
    tools=[ngo_lookup],
    model=llm_model,
)

relief_agent = Agent(
    name="Relief Program Agent",
    handoff_description="Disaster relief, cash/food/NFI distribution, shelter support.",
    instructions=(
        "You are an NGO Disaster Relief officer focused on Pakistan. Explain eligibility, registration process, distribution points, "
        "helpline numbers, and verification needed for disaster assistance (floods, earthquakes, heat waves). "
        "Where helpful, reference operational orgs like Edhi and Chhipa (ambulance & emergency relief), "
        "Saylani (ration/food distribution), and local district administration protocols."
    ),
    tools=[ngo_lookup],
    model=llm_model,
)

livelihood_agent = Agent(
    name="Livelihoods Program Agent",
    handoff_description="Skills training, micro-grants, job placement support.",
    instructions=(
        "You are an NGO Livelihoods officer in Pakistan. Provide details on vocational training, micro-grants, "
        "entrepreneurship support, and job placement guidance. Reference Akhuwat (interest-free microfinance, small-business grants), "
        "Saylani (IT/digital skills), and provincial skills development programs when relevant. Include application steps and timelines."
    ),
    tools=[ngo_lookup],
    model=llm_model,
)

protection_agent = Agent(
    name="Protection & Safeguarding Agent",
    handoff_description="GBV/child protection referrals, case management intake info.",
    instructions=(
        "You are an NGO Protection focal person in Pakistan. Provide trauma-informed, survivor-centered information. "
        "Explain safe referral pathways (e.g., government helplines and reputable NGOs), child protection protocols, and emergency contacts. "
        "Avoid collecting sensitive details in chat and guide users to secure, confidential channels."
    ),
    tools=[ngo_lookup],
    model=llm_model,
)

volunteer_agent = Agent(
    name="Volunteer Management Agent",
    handoff_description="Volunteer sign-up, onboarding, training schedules.",
    instructions=(
        "You manage volunteers across Pakistan-based NGOs. Explain how to register, screening steps, orientation, and training schedules. "
        "Where relevant, reference volunteer pathways with Edhi (rescue/relief), Saylani (ration drives/IT training), and local hospital or school-based initiatives. "
        "Do not store personal identifiers in chat; direct users to official sign-up forms if needed."
    ),
    tools=[ngo_lookup],
    model=llm_model,
)

complaints_agent = Agent(
    name="Complaints & Feedback Agent",
    handoff_description="Grievance redressal mechanism (GRM), accountability, hotlines.",
    instructions=(
        "You handle complaints and feedback. Explain the GRM process, confidentiality, and escalation steps. "
        "In Pakistan context, describe how to raise concerns via NGO hotlines, email forms, or in-person desks, and clarify expected response times. "
        "Do not collect sensitive identifiers in chat; guide to official complaint forms or hotlines."
    ),
    tools=[ngo_lookup],
    model=llm_model,
)


# Triage Agent (with input guardrail)
async def ngo_input_guardrail(ctx, agent, input_data):
    result = await Runner.run(guardrail_agent, input_data, context=ctx.context)
    flags = result.final_output_as(NGOInputCheck)

    trip = False
    reasons = []

    if flags.unsafe_request:
        trip = True
        reasons.append("Unsafe content (violence/self-harm/hate).")
    if flags.collect_personal_data:
        trip = True
        reasons.append("Attempt to share/store personal identifiers without consent workflow.")
    if flags.medical_diagnosis:
        trip = True
        reasons.append("Medical diagnosis/prescription request.")
    if flags.legal_advice:
        trip = True
        reasons.append("Request for binding legal advice.")

    # Keep topic in context metadata for triage
    ctx.context["ngo_topic"] = flags.topic

    return GuardrailFunctionOutput(
        output_info=flags,
        tripwire_triggered=trip,
        tripwire_message="; ".join(reasons) if reasons else "",
    )

triage_agent = Agent(
    name="NGO Triage Agent",
    instructions=(
        "Decide the single best-matching category for the user's request. "
        "Respond with exactly one label from: education, health, relief, livelihood, protection, volunteering, complaints, general."
    ),
    # input_guardrails=[InputGuardrail(guardrail_function=ngo_input_guardrail)],
    model=llm_model,
)

# Fallback agar handoff fail ho jaye
@function_tool
def default_reply(user_query: str) -> str:
    return f"Sorry, I couldn’t find exact NGO info for: {user_query}. Please try with categories like education, health, relief, livelihood, protection."



# Routing
CATEGORY_TO_AGENT = {
    "education": education_agent,
    "health": health_agent,
    "relief": relief_agent,
    "livelihood": livelihood_agent,
    "protection": protection_agent,
    "volunteering": volunteer_agent,
    "complaints": complaints_agent,
    # fallback for "general": route to education agent (could be changed to a dedicated info-desk agent)
    "general": education_agent,
}

welcome_text = (
    "Welcome to the Pakistan NGO Assistant!\n\n"
    "Ask about:\n"
    "• Education (scholarships, enrollment)\n"
    "• Health (medical/vaccination camps)\n"
    "• Relief (flood/earthquake assistance)\n"
    "• Livelihoods (skills training, micro-grants)\n"
    "• Protection (safe referrals)\n"
    "• Volunteering (join & trainings)\n"
    "• Donations (secure channels & receipts)\n"
    "• Complaints (GRM & hotlines)\n\n"
    "For your safety, do not share CNIC/phone/address here. We'll direct you to official forms when needed."
)

@cl.on_chat_start
async def on_chat_start():
    await cl.Message(content=welcome_text).send()

@cl.on_message
async def on_message(message: cl.Message):
    try:
        # Step 1: Triage with guardrails
        await cl.Message(content=f"**Triage** is analyzing your request: \"{message.content}\" ").send()
        triage_result = await Runner.run(triage_agent, message.content)

        # Prefer explicit category from triage; else use guardrail topic; else fallback to general
        category = (triage_result.final_output or "").strip().lower()
        topic_from_guardrail = triage_result.context.get("ngo_topic") if hasattr(triage_result, "context") else None
        category = category or (topic_from_guardrail or "general")

        chosen_agent = CATEGORY_TO_AGENT.get(category, education_agent)

        # Step 2: Announce handoff
        await cl.Message(content=f"Handoff → **{chosen_agent.name}** (category: {category})").send()

        # Step 3: Get program response
        program_result = await Runner.run(chosen_agent, message.content)
        await cl.Message(content=f"**{chosen_agent.name}**:\n{program_result.final_output}").send()

    except InputGuardrailTripwireTriggered as e:
        reason = getattr(e, "args", ["Guardrail activated"])[0]
        safety_note = (
            "**Guardrail Activated**\n"
            f"Reason: {reason}\n\n"
            "You can still ask about our programs listed above. "
            "For personal data submission, medical concerns, or legal matters, "
            "we'll direct you to the correct official channels."
        )
        await cl.Message(content=safety_note).send()

Dockerfile

In [ ]:
# Use a specific and recent Python slim image for reproducibility and efficiency.
FROM python:3.10.13-slim

# The working directory is now set to the root of the container.
WORKDIR /

# Copy the requirements file first to optimize Docker's build cache.
COPY requirements.txt .

# Install dependencies from requirements.txt and also install openai-agents directly.
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt && \
    pip install --no-cache-dir openai-agents

# Copy the rest of your files directly into the container's root.
COPY . .

# Create and set permissions for necessary files/directories.
RUN mkdir -p .files .chainlit && \
    touch chainlit.md

# Create a non-root user for security.
RUN adduser --disabled-password --gecos "" chainlit
RUN chown -R chainlit:chainlit .files .chainlit chainlit.md
USER chainlit

# Set Chainlit environment variables.
ENV CHAINLIT_UI=True
ENV CHAINLIT_BROWSER_AUTO_OPEN=false

# Expose the port on which the application will run.
EXPOSE 7860

# Command to run the application.
CMD ["chainlit", "run", "app.py", "--host", "0.0.0.0", "--port", "7860"]

requirements.txt

In [ ]:
chainlit
python-dotenv
websockets
openai-agents

Variable Name in settings= CHAINLIT_RUN_MODULE\
app

Licence= Apache 2.0 license